In [1]:
# still need to make some improvements like edge cases for better fool proof
import sys
import torch
import torch.nn as nn
import torch.nn.functional as FF

import torch.nn.functional as F
project_root = "/Users/sanedara/Documents/Machine_learning_practise/recommendation_system"
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from src.data.loader import MovieLensLoader
import pandas as pd
import numpy as np

In [2]:
loader = MovieLensLoader()
users = loader.load_users()
movies = loader.load_movies()
ratings = loader.load_ratings().sort_values('timestamp')
movies.head()

,movie_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Childrens,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [3]:
# initialize a random user and item embedding matrix
# we train these embeddings using BPR for score calculation and Loss function
def split_user(df, frac=0.85):
    df = df.sort_values("timestamp")
    cut = int(frac * len(df))
    return df.iloc[:cut], df.iloc[cut:]


# split data in to train and test, 85% and 15% respectively
def train_test_split():
    train_parts = []
    test_parts = []
    for uid, df_u in ratings.groupby("user_id"):
        if len(df_u) < 15:
            continue
        tr, te = split_user(df_u, 0.85)
        if len(tr) == 0 or len(te) == 0:
            continue
        train_parts.append(tr)
        test_parts.append(te)
    
    train_split = pd.concat(train_parts, ignore_index=True)
    test_split  = pd.concat(test_parts, ignore_index=True)
    return train_split, test_split



# we build this only for train since we dont use test for training and dont need these conversations
def build_id_maps(train_df):
    uidx = np.sort(train_df['user_id'].unique())
    midx = np.sort(train_df['item_id'].unique())

    uid2idx = {id_: i for i, id_ in enumerate(uidx)}
    mid2idx = {id_:i for i, id_ in enumerate(midx)}
    midx2id = {i:id_ for id_, i in mid2idx.items()}
    return uid2idx, mid2idx, midx2id


def get_user_sets(train_df, uid2idx, mid2idx):
    u_seen_m, u_pos_m = {}, {}
    for uid, df in train_df.groupby('user_id'):
        uid = uid2idx[uid]
        m_seen = set([mid2idx[mid] for mid in df['item_id'].values if mid in mid2idx])
        
        m_pos = df[df['rating'] >= 4]['item_id'].values
        m_pos_idx = [mid2idx[mid] for mid in m_pos if mid in mid2idx]
        if len(m_pos_idx) == 0:
            continue
        u_seen_m[uid] = m_seen
        u_pos_m[uid] = m_pos_idx
    return u_seen_m, u_pos_m

# u_seen, u_pos = get_user_sets(train_parts)
    


In [4]:
class DatasetProcessing():
    def __init__(self, u_seen, u_pos, num_movies, batch_size):
        self.batch_size = batch_size
        self.u_seen = u_seen
        self.u_pos = u_pos
        self.users = np.array(list(self.u_seen.keys()), dtype=np.int64)
        self.movies = num_movies
        self.rnd = np.random.default_rng(seed=42)

    def prepare(self, N = 5,M = 12):
        usr = self.rnd.choice(self.users, size=self.batch_size, replace=True)

        i_pos = np.empty((self.batch_size, N), dtype=np.int64)
        # i_neg = np.empty(self.batch_size, dtype=np.int64)
        i_neg = np.empty((self.batch_size, M), dtype=np.int64)
        
        for t, u in enumerate(usr):
            _pos = self.u_pos[u]
            # i_pos[t] = _pos[self.rnd.integers(0, len(_pos))]
            p_sampled = []
            while len(p_sampled) < N:
                j = _pos[self.rnd.integers(0, len(_pos))]
                p_sampled.append(j)
            i_pos[t] = p_sampled

            # _neg = self.rnd.integers(0, self.movies)
            # while _neg in _seen:
            #     _neg = self.rnd.integers(0, self.movies)
            # i_neg[t] = _neg
            
            _seen = self.u_seen[u]
            n_sampled = []
            while len(n_sampled) < M:
                j = self.rnd.integers(0, self.movies)
                if j not in _seen:
                    n_sampled.append(j)
            i_neg[t] = n_sampled
            

        return (
            torch.tensor(usr, dtype=torch.long),
            torch.tensor(i_pos, dtype=torch.long),
            torch.tensor(i_neg, dtype=torch.long)
        )

# dp = DatasetProcessing(u_seen, u_pos, len(movies), 32)
# genereate random user and item embeddings, train these embeddings using BPR 

class BPRModel(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim):
        super().__init__()
        self.u_emb = nn.Embedding(n_users, emb_dim)
        self.m_emb = nn.Embedding(n_movies, emb_dim)
        self.u_bias = nn.Embedding(n_users, 1)
        self.m_bias = nn.Embedding(n_movies, 1)

        # normalize
        nn.init.normal_(self.u_emb.weight, std=0.01)
        nn.init.normal_(self.m_emb.weight, std=0.01)
        nn.init.normal_(self.u_bias.weight, std=0.01)
        nn.init.normal_(self.m_bias.weight, std=0.01)

    def score(self, u, m):
        u_e = self.u_emb(u)
        m_e = self.m_emb(m)
        s = (u_e * m_e).sum(dim=1) + (self.u_bias(u).squeeze(1)) + (self.m_bias(m).squeeze(1))
        return s



In [5]:
def train(model, _data, device, reg_lambda: float = 1e-4):
    loss = 0
    _uids, _pids, _nids = _data.prepare()
    _uids = _uids.to(device)
    _pids = _pids.to(device)
    _nids = _nids.to(device)

    B, N = _pids.shape
    uids_rep_pos = _uids.repeat_interleave(N)     # (B*N,)
    pids_flat = _pids.reshape(-1)             # (B*N,)
    # flatten negatives to score in one call
    B, M = _nids.shape
    uids_rep_neg = _uids.repeat_interleave(M)     # (B*M,)
    nids_flat = _nids.reshape(-1)             # (B*M,)
    
    
    # ps = model.score(_uids, _pids)
    ps = model.score(uids_rep_pos, pids_flat).view(B, N)  # (B, N)
    ns = model.score(uids_rep_neg, nids_flat).view(B, M)  # (B, M)
    # ns = model.score(_uids, _nids)
    # x = ps - ns
    x = ps.unsqueeze(2) - ns.unsqueeze(1)   # (B, N, M)
    loss = F.softplus(-x).mean()  # stable: log(1 + exp(-(s_pos - s_neg)))

    # L2 regularization on embeddings used in this batch
    pu = model.u_emb(_uids)
    qi = model.m_emb(_pids)
    qj = model.m_emb(_nids)
    # reg = (pu.pow(2).sum(dim=1) + qi.pow(2).sum(dim=1) + qj.pow(2).sum(dim=1)).mean()
    # loss = loss + reg_lambda * reg
    # reduce each term to (B,) so they can be added
    pu_reg = pu.pow(2).sum(dim=1)                 # (B,)
    qi_reg = qi.pow(2).sum(dim=2) .mean(dim=1)    # (B,)
    qj_reg = qj.pow(2).sum(dim=2).mean(dim=1)     # (B,)  <-- sum over D, mean over M
    
    reg = (pu_reg + qi_reg + qj_reg).mean()       # scalar
    loss = loss + reg_lambda * reg
    return loss

train_parts, test_parts = train_test_split()
uid2idx, mid2idx, midx2id = build_id_maps(train_parts)
u_seen, u_pos = get_user_sets(train_parts, uid2idx, mid2idx)

n_users = len(uid2idx.keys())
n_movies = len(mid2idx.keys())

batch_size = 60
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Mac GPU)")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU")
    
_data = DatasetProcessing(u_seen, u_pos, n_movies, batch_size)
model = BPRModel(
    n_users = n_users,
    n_movies = n_movies,
    emb_dim = 50,
)
model = model.to(device)
opt = torch.optim.Adam(model.parameters(),lr=1e-03)


epochs, steps = 15, 10000
for e in range(epochs):
    total = 0
    for s in range(steps):
        opt.zero_grad()
        loss = train(model, _data, device)
        loss.backward()
        opt.step()
        total += float(loss.item())
    print(f"Epoch {e}/{epochs}  avg_loss={total/steps:.4f}")

Using MPS (Mac GPU)
Epoch 0/15  avg_loss=0.1033
Epoch 1/15  avg_loss=0.0254
Epoch 2/15  avg_loss=0.0163
Epoch 3/15  avg_loss=0.0134
Epoch 4/15  avg_loss=0.0120
Epoch 5/15  avg_loss=0.0113
Epoch 6/15  avg_loss=0.0107
Epoch 7/15  avg_loss=0.0104
Epoch 8/15  avg_loss=0.0101
Epoch 9/15  avg_loss=0.0099
Epoch 10/15  avg_loss=0.0098
Epoch 11/15  avg_loss=0.0096
Epoch 12/15  avg_loss=0.0095
Epoch 13/15  avg_loss=0.0094
Epoch 14/15  avg_loss=0.0093


In [6]:
@torch.no_grad()
def recommend_topk(model: BPRModel, user_id: int, uid2idx: dict, idx2iid: dict, seen_by_u: dict, k: int = 12, device: str = "cpu"):
    if user_id not in uid2idx:
        return []
    u = uid2idx[user_id]
    seen = seen_by_u.get(u, set())

    # score all items
    num_items = model.m_emb.num_embeddings
    u_tensor = torch.full((num_items,), u, dtype=torch.long, device=device)
    i_tensor = torch.arange(num_items, dtype=torch.long, device=device)

    scores = model.score(u_tensor, i_tensor).cpu().numpy()

    # remove seen by setting score very low
    if seen:
        scores[list(seen)] = -1e18

    top_idx = np.argpartition(-scores, k)[:k]
    top_idx = top_idx[np.argsort(-scores[top_idx])]

    # map back to original item_id
    return [idx2iid[int(i)] for i in top_idx]
k=12
recs = recommend_topk(model, user_id=10, uid2idx=uid2idx, idx2iid=midx2id, seen_by_u=u_seen, k=12, device=device)
def convert_mids_to_titles(recs):
    titles = movies.loc[movies["movie_id"].isin(recs), ["movie_id", "title"]]
    # keep the same order as recs
    titles = titles.set_index("movie_id").loc[recs].reset_index()
    print(titles)
convert_mids_to_titles(recs)

    movie_id                                   title
0         79                    Fugitive, The (1993)
1        187          Godfather: Part II, The (1974)
2        202                    Groundhog Day (1993)
3        607                          Rebecca (1940)
4        318                 Schindler's List (1993)
5        443                       Birds, The (1963)
6         58                        Quiz Show (1994)
7        491    Adventures of Robin Hood, The (1938)
8        507        Streetcar Named Desire, A (1951)
9        177  Good, The Bad and The Ugly, The (1966)
10       238                  Raising Arizona (1987)
11       648                   Quiet Man, The (1952)


In [7]:
def build_test_positives(test_df, min_rating=4):
    test_pos = (
        test_df[test_df["rating"] >= min_rating]
        .groupby("user_id")["item_id"]
        .apply(set)
        .to_dict()
    )
    return test_pos

import numpy as np

def recall_at_k(recs, gt_set, k):
    recs_k = recs[:k]
    if len(gt_set) == 0:
        return None
    hits = sum(1 for i in recs_k if i in gt_set)
    return hits / len(gt_set)

def ndcg_at_k(recs, gt_set, k):
    recs_k = recs[:k]
    if len(gt_set) == 0:
        return None

    dcg = 0.0
    for rank, item in enumerate(recs_k, start=1):  # rank starts at 1
        if item in gt_set:
            dcg += 1.0 / np.log2(rank + 1)

    m = min(k, len(gt_set))
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, m + 1))

    return dcg / idcg if idcg > 0 else 0.0


In [11]:
def evaluate(model, test_df, uid2idx, midx2id, seen_by_u, k, device):
    test_pos_by_user = build_test_positives(test_df, min_rating=4)

    recalls = []
    ndcgs = []

    for user_id, gt_set in test_pos_by_user.items():
        # only evaluate users the model knows
        if user_id not in uid2idx:
            continue

        recs = recommend_topk(
            model,
            user_id=user_id,
            uid2idx=uid2idx,
            idx2iid=midx2id,
            seen_by_u=seen_by_u,
            k=k,
            device=device,
        )

        r = recall_at_k(recs, gt_set, k)
        n = ndcg_at_k(recs, gt_set, k)

        if r is not None:
            recalls.append(r)
        if n is not None:
            ndcgs.append(n)

    return float(np.mean(recalls)), float(np.mean(ndcgs)), len(recalls)
K = 24
recall, ndcg, n_users = evaluate(model, test_parts, uid2idx, midx2id, u_seen, K, device)
print(f"Users evaluated: {n_users}")
print(f"Recall@{K}: {recall:.4f}")
print(f"NDCG@{K}: {ndcg:.4f}")


Users evaluated: 898
Recall@24: 0.2267
NDCG@24: 0.1584
